# DTSC 520 - Environment Check

**Run this before the course needs it, not on the night it breaks.**

Kernel -> Restart & Run All. It takes a few seconds and changes nothing on your
machine - there are no installers here and nothing is repaired. It only looks.

The last cell prints a short report. If something fails, copy that report into
your email to me and the problem is usually solvable in one reply. Without it,
"it doesn't work" is not something anyone can act on.


## The checks

Each one prints its own result as it runs.

In [ ]:
import sys, platform, importlib, os, io, traceback

REPORT = []      # (name, status, detail) - status is "ok", "note" or "FAIL"

# Not every assignment needs every library. Module 2 is pure Python; numpy
# arrives in Module 3, pandas in Module 4, matplotlib in Module 5. Reporting a
# red FAIL because a student has not yet installed something the course has not
# yet asked for would be alarming and wrong - and the students most likely to
# run this are exactly the ones least able to tell a real problem from a
# premature one. Anything optional that is missing is reported as a NOTE, and
# does not count against the result.
class Skip(Exception):
    """Raised by a check that cannot run yet, and should not count as failure."""


def check(name, required=True):
    def wrap(fn):
        try:
            ok, detail = fn()
            status = "ok" if ok else ("FAIL" if required else "note")
        except Skip as e:
            status, detail = "note", str(e)
        except Exception as e:
            detail = f"{type(e).__name__}: {e}"
            status = "FAIL" if required else "note"
        REPORT.append((name, status, detail))
        print(f"  {status:6s}{name}: {detail}")
        return fn
    return wrap


MISSING = []     # libraries to offer an install command for at the end


def installer(pkg):
    """The install command that targets THIS kernel, not some other Python.

    `%pip` and `%conda` are the notebook magics, and they matter here: plain
    `pip install` in a terminal installs into whichever Python is first on the
    PATH, which on a machine with two Pythons is exactly the one Jupyter is not
    running. That is the same fault this notebook checks for two cells earlier,
    so recommending the command that causes it would be perverse.
    """
    if os.environ.get("CONDA_PREFIX") or "conda" in sys.version.lower():
        return f"%conda install -y {pkg}"
    return f"%pip install {pkg}"


def have(lib):
    """Import a library, or Skip with wording that does not read as a fault."""
    try:
        return importlib.import_module(lib)
    except ImportError:
        raise Skip(f"{lib} is not installed yet - see the install line above")


print("Running checks. Nothing here modifies your machine.\n")


In [ ]:
@check("Python version")
def _():
    v = sys.version_info
    ok = (v.major, v.minor) >= (3, 9)
    return ok, f"{platform.python_version()} ({'fine' if ok else 'too old - need 3.9+'})"


@check("Jupyter is using this Python")
def _():
    # The classic invisible fault: Python installed twice, pip installing into
    # one and Jupyter running the other. Nothing looks wrong until an import
    # fails for a package you know you installed.
    return True, sys.executable


In [ ]:
# from_module says when the course first needs each one, so a "note" reads as
# "not yet" rather than "broken".
for lib, need, from_module in [("numpy", "1.20", "Module 3"),
                               ("pandas", "1.3", "Module 4"),
                               ("matplotlib", "3.4", "Module 5")]:
    @check(f"{lib} installed", required=False)
    def _(lib=lib, need=need, from_module=from_module):
        try:
            m = importlib.import_module(lib)
        except ImportError:
            MISSING.append(lib)
            raise Skip(f"not installed. Not needed until {from_module}, but "
                       f"there is no reason to wait - run  {installer(lib)}  "
                       f"in a cell here, then re-run this notebook")
        v = getattr(m, "__version__", "unknown")
        return True, f"version {v} (course was built against {need}+)"


In [ ]:
@check("read_csv from a relative path", required=False)
def _():
    # This is the one that actually catches people. Module 4 reads
    # `data/...` relative to the notebook; if Jupyter was started from a
    # different directory, that raises FileNotFoundError and it reads like a
    # missing file rather than a working-directory problem.
    pd = have("pandas")
    os.makedirs("data", exist_ok=True)
    probe = os.path.join("data", "_env_check_probe.csv")
    with open(probe, "w") as fh:
        fh.write("student_id,gpa\n1,3.4\n2,3.9\n")
    df = pd.read_csv(probe)
    os.remove(probe)
    try:
        os.rmdir("data")          # only if we created it and it is now empty
    except OSError:
        pass
    return len(df) == 2, f"read 2 rows from {probe!r}; working directory is {os.getcwd()}"


In [ ]:
@check("matplotlib draws inline", required=False)
def _():
    matplotlib = have("matplotlib")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(3, 1.4))
    ax.plot([1, 2, 3], [2, 1, 3])
    ax.set_title("if you can see this line, plotting works")
    plt.show()
    plt.close(fig)
    return True, f"backend {matplotlib.get_backend()}"


In [ ]:
@check("%%expect can register")
def _():
    # Every deliberate-error cell in Modules 2-4 depends on this magic. But it
    # is registered by each module's setup cell, which this standalone notebook
    # never runs - so asking "is it registered?" would report FAIL on a
    # perfectly healthy machine, which is worse than not checking at all.
    #
    # The question that matters is whether the MACHINERY works here. So do what
    # the setup cell does, on a throwaway name, and confirm it took.
    from IPython import get_ipython
    from IPython.core.magic import register_cell_magic
    ip = get_ipython()
    if ip is None:
        return False, "not running under IPython/Jupyter - open this in Jupyter"

    @register_cell_magic
    def _dtsc520_probe(line, cell):
        return None

    ok = "_dtsc520_probe" in ip.magics_manager.magics.get("cell", {})
    ip.magics_manager.magics.get("cell", {}).pop("_dtsc520_probe", None)
    return ok, ("cell magics register correctly, so %%expect will work once "
                "the module setup cell runs" if ok else
                "cell magics could not be registered")


## Your report

This is the bit to copy.

In [ ]:
ok    = sum(1 for _, st, _ in REPORT if st == "ok")
notes = sum(1 for _, st, _ in REPORT if st == "note")
bad   = sum(1 for _, st, _ in REPORT if st == "FAIL")

if bad:
    verdict = f"{bad} problem(s) to fix - see the FAIL lines below"
elif notes:
    verdict = f"good. {notes} thing(s) not installed yet, which is fine for now"
else:
    verdict = "all good"

lines = [
    "----- DTSC 520 environment report -----",
    f"verdict     : {verdict}",
    f"checks      : {ok} ok, {notes} note, {bad} fail",
    f"python      : {platform.python_version()}",
    f"executable  : {sys.executable}",
    f"platform    : {platform.platform()}",
    f"working dir : {os.getcwd()}",
    "",
]
for name, st, detail in REPORT:
    lines.append(f"[{st:4s}] {name}: {detail}")
lines.append("---------------------------------------")

print("\n".join(lines))
print()
if bad:
    print("Copy everything between the dashed lines into your email.")

if MISSING:
    # One command for everything missing, rather than making them run three.
    # Offered, not executed - this notebook never changes your machine.
    print("You are missing: " + ", ".join(MISSING))
    print()
    print("You do not need them yet, but installing now means one less thing to")
    print("go wrong later. Make a new cell below and run:")
    print()
    print("    " + installer(" ".join(MISSING)))
    print()
    print("Then Kernel -> Restart & Run All to confirm it worked.")
elif not bad:
    print("Nothing to do - everything this course needs is already here.")
